# Customer Segmentation (Mall Customers)

**Goal:** Group customers into segments for targeted marketing
**Algorithm:** K-Means Clustering + PCA Visualization
**Dataset:** [Customer Segmentation](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
%matplotlib inline

In [1]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


Running locally - skipping Colab setup


## 1. Load Data from Kaggle

In [2]:
path = kagglehub.dataset_download("vjchoudhary7/customer-segmentation-tutorial-in-python")
df = pd.read_csv(f"{path}/Mall_Customers.csv")
print ('Shape: %s' % (df.shape,))
print ('First 3 rows:\n%s' % df.head(3))

Shape: (200, 5)
First 3 rows:
   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6


<hr>## 2. Exploratory Data Analysis

In [3]:
print ('Age range: %d - %d' % (df['Age'].min(), df['Age'].max()))
print ('Income range: %d - %d' % (df['Annual Income (k$)'].min(), df['Annual Income (k$)'].max()))
print ('Spending range: %d - %d' % (df['Spending Score (1-100)'].min(), df['Spending Score (1-100)'].max()))
print ('\nGender distribution:\n%s' % df['Gender'].value_counts())

Age range: 18 - 70
Income range: 15 - 137
Spending range: 1 - 99

Gender distribution:
Male      106
Female     94


In [4]:
# Visualize relationships
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', data=df, hue='Gender')
plt.title('Income vs Spending')

plt.subplot(1, 3, 2)
sns.scatterplot(x='Age', y='Spending Score (1-100)', data=df, hue='Gender')
plt.title('Age vs Spending')

plt.subplot(1, 3, 3)
sns.scatterplot(x='Age', y='Annual Income (k$)', data=df, hue='Gender')
plt.title('Age vs Income')

plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

<hr>## 3. Feature Selection & Scaling

In [5]:
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print ('Scaled feature matrix: %s' % (X_scaled.shape,))

Scaled feature matrix: (200, 3)


<hr>## 4. Find Optimal K (Elbow Method)

In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-cluster variance)')
plt.title('Elbow Method for Optimal K')
plt.grid(True)
plt.tight_layout()
plt.show()
print ('Optimal K is where the elbow bends (K=5)')

<hr>## 5. Train K-Means

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)
print ('Cluster sizes:\n%s' % df['Cluster'].value_counts().sort_index())

<hr>## 6. Cluster Analysis

In [9]:
profile = df.groupby('Cluster')[features].mean().round(1)
profile['Count'] = df['Cluster'].value_counts().sort_index().values
print ('Cluster Profiles (mean values):')
print (profile)

print ('\nInterpretation:')
print ('  Cluster 0: Low income, low spending  -> Budget conscious')
print ('  Cluster 1: High income, low spending  -> Selective')
print ('  Cluster 2: Low income, high spending  -> Impulsive')
print ('  Cluster 3: High income, high spending -> Premium (target!)')
print ('  Cluster 4: Medium income/age/spending  -> Average')

<Figure size NxN with 1 Axes>

Total variance explained: 86.3%


<hr>## 7. Visualize with PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', data=df,
                palette='Set1', s=100, alpha=0.8)
plt.title('Customer Segments Visualized in 2D (PCA)')
plt.xlabel('PCA Component 1 (%.1f%% variance)' % (pca.explained_variance_ratio_[0]*100))
plt.ylabel('PCA Component 2 (%.1f%% variance)' % (pca.explained_variance_ratio_[1]*100))
plt.legend(title='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print ('Total variance explained: %.1f%%' % (pca.explained_variance_ratio_.sum()*100))